In [ ]:
import numpy as np
from scipy.special import genlaguerre
from scipy.stats import qmc, skew
import matplotlib.pyplot as plt
import polarTransform 
from scipy.differentiate import derivative 
import scipy.integrate as integrate

In [ ]:
#Define functions for phase mismatch and effective refractive indices as a function of theta (angle between pump and crystal axis)
def sctmp(no,ne,th):
    return np.square(no*np.sin(th))+np.square(ne*np.cos(th)) 

def eta(no,ne,th):
    return no*ne/np.sqrt(sctmp(no,ne,th))
    
def no(lamda):  #ordinary index in BBO, lambda is in microns
    return np.sqrt(2.7405+0.0184/(lamda**2-0.0179)-0.0155*lamda**2)
    
def ne(lamda):   #extraordinary index in BBO, lambda is in microns
    return np.sqrt(2.3730+0.0128/(lamda**2-0.0156)-0.0044*lamda**2)

def cart_transform(u, gridpts=256):  # Polar -> Cartesian
    cartu, _ = polarTransform.convertToCartesianImage(
        np.real(u), imageSize=(gridpts, gridpts)
    )
    return cartu

#Compute refractive indices
npo = no(0.355)
npe = ne(0.355)
nso = no(0.710)

In [ ]:

# --- Phase Mismatch Term---
def delta_kz(qs, qi, phis, phii, theta):
    thp = np.radians(theta)
    q_sum2 = qs**2 + qi**2 + 2 * qs * qi * np.cos(phis - phii)
    k_p = 2.0 * np.pi * eta(npo, npe, thp) / 0.355
    k_si = 2.0 * np.pi * nso / 0.710
    walk_off_angle = 0*4.6/ 180 * np.pi
    return np.sqrt(k_p**2 - q_sum2) - np.sqrt(k_si**2 - qs**2) - np.sqrt(k_si**2 - qi**2) + np.tan(walk_off_angle) * (qs * np.cos(phis) + qi * np.cos(phii)) 

# --- Pump Spatial Profile Term---
def pump_profile4d(qs, qi, phis, phii, wp, lp, pp):
    q_sum2 = qs**2 + qi**2 + 2 * qs * qi * np.cos(phis - phii)
    if lp == 0 and pp == 0:
        return np.exp(-wp**2 * q_sum2 / 4)
    else:
        xi = np.arctan2(qs * np.cos(phis)+ qi * np.cos(phii), qs * np.sin(phis)+ qi * np.sin(phii))
        return genlaguerre(pp, abs(lp))(q_sum2 * wp**2 / 2) * np.exp(-q_sum2 * wp**2 / 4) * np.sqrt(q_sum2)**np.abs(lp) * np.exp(1j * lp * xi)

# --- Pump Spatial Profile Term---
# L = length of the nonlinear crystal [um]; wp = waist diameter of the pump [um]; theta = angle between optical axis and propagation [deg]
# lp = pump oam index; pp = pump radial index
def oam_decompose(L = 3000, lp = 0, pp = 0, wp = 80, theta = 32.910):
    # --- Parameters ---
    wavelength_p = 0.355  # um
    wavelength_s = 0.710  # um
    thp = np.radians(theta)
    
    k_p = 2.0 * np.pi * eta(npo, npe, thp) / 0.355
    k_si = 2.0 * np.pi * nso / 0.710
    
    # --- Grid ---
    qmax = 0.12 # 1/um
    L_max = 30 #Maximum OAM range covered
    gridpts = 64 #Use more grid points for finer resolution, but keep an eye on the computation resources
    qs, qi, phis, phii = np.meshgrid(
                np.linspace(1e-3, 2.0 * qmax, gridpts, endpoint = False),
                np.linspace(1e-3, 2.0 * qmax, gridpts, endpoint = False),
                np.linspace(0, 2.*np.pi, gridpts, endpoint = False),
                np.linspace(0, 2.*np.pi, gridpts, endpoint = False),
                indexing='ij'
            )

    # --- Pump Envelope ---
    q_sum2 = qs**2 + qi**2 + 2 * qs * qi * np.cos(phis - phii)

    alpha_q = pump_profile4d(qs, qi, phis, phii, wp, lp, pp)


    # --- Joint Wavefunction ---
    dkz = delta_kz(qs, qi, phis, phii, theta)
    phase_match = np.sinc(dkz * L / (2 * np.pi)) * np.exp(1j * dkz * L / 2)
    
    
    Psi = alpha_q * phase_match #Construct two photon wavefunction = pump_profile x phase matching term

    tr_Psi_q = np.sum(np.sum(Psi * qs * qi, axis = 0), axis = 0) #Trace out signal and idler radial coordinates for OAM spectrum computation

    tr_Psi_sig = np.sum(np.sum(Psi * qs, axis = 0), axis = -2) #Trace out the idler radial and angular coordinates for far-field intensity output

    l_vals = np.arange(-L_max, L_max + 1) 
    Gamma = np.zeros((len(l_vals), len(l_vals)), dtype=complex)
    size = 2 * L_max + 1    

    phis_1d = np.linspace(0, 2.*np.pi, gridpts, endpoint = False)
    phii_1d = np.linspace(0, 2.*np.pi, gridpts, endpoint = False)


# Psi_q should be shape (gridpts, gridpts) with axes (phis, phii)
# build coefficient matrix
    C = np.zeros((len(l_vals), len(l_vals)), dtype=complex)
    for i, l_i in enumerate(l_vals):
        for j, l_s in enumerate(l_vals):
            C[j, i] = np.sum(
                tr_Psi_q
                * np.exp(-1j * l_s * phis_1d[:, None])
                * np.exp(-1j * l_i * phii_1d[None, :])
            ) 

   
    oam_spec = np.abs(C)**2
    oam_spec /= oam_spec.sum() #Normalize
    return oam_spec, tr_Psi_sig #Output the OAM spectrum and the signal far-field intensity

In [ ]:
S, sig_amp = oam_decompose(L = 2000, lp = 0, pp = 0, wp = 200, theta = 32.915)

In [ ]:
sig_img_cart = cart_transform(np.transpose(np.abs(sig_amp)**2), gridpts=128)
plt.xlabel(r'$q_x$ (mrad)',font = 'Helvetica', size = 20)
plt.ylabel(r'$q_y$ (mrad)',font = 'Helvetica', size = 20)
plt.xticks([20, 42, 64, 86, 108], [-20, -10, 0, 10, 20])
plt.yticks([20, 42, 64, 86, 108], [-20, -10, 0, 10, 20])
plt.tick_params(axis='both', labelsize=18)
plt.imshow(sig_img_cart)